# Модель 2, шаг 2 — применение эмбеддингов (FAISS top-N)

Строит FAISS-индекс по корпусу объявлений и находит top-N (по умолчанию 300 —
с запасом, не 50, чтобы у LightGBM-ре-ранкера (модель 3) было из чего
выбирать) ближайших по косинусной близости объявлений для каждого запроса.

**Важно:** если собираете полный ансамбль (модели 1+2+3), этот ноутбук нужно
прогнать **дважды** с разным конфигом:

1. `QUERIES_FILE/ITEMS_FILE = train_queries.parquet/train_items.parquet`,
   `QUERY_ID_COL = "query_uid"`, `OUT_PATH = "./embed_candidates_train.parquet"`,
   `CACHE_DIR = "./retrieve_cache_train"` — для обучения ре-ранкера
   (`03_train_reranker.ipynb`).
2. `QUERIES_FILE/ITEMS_FILE = benchmark_queries.parquet/benchmark_items.parquet`,
   `QUERY_ID_COL = "query_id"`, `OUT_PATH = "./embed_candidates.parquet"`,
   `CACHE_DIR = "./retrieve_cache_benchmark"` — для финального прогона
   (`generate_final_answer.py`).

Промежуточные результаты (эмбеддинги, результаты FAISS-поиска) кэшируются на
диск в `CACHE_DIR` — если ноутбук упадёт (нехватка памяти, краш VS Code) после
какого-то шага, повторный запуск подхватит уже посчитанное из кэша вместо
пересчёта с нуля.

In [10]:
import os
import time
import numpy as np
import pandas as pd
import faiss
import torch
from sentence_transformers import SentenceTransformer

from text_prep import build_embedding_query_text, build_embedding_item_text

## Конфиг — поправьте под себя

In [11]:
BASE = "."                            # папка с parquet-файлами
MODEL_PATH = "./biencoder_finetuned"
BASE_MODEL_NAME = "intfloat/multilingual-e5-base"  # тот BASE_MODEL, что был в 01_train_biencoder.ipynb -
                                                     # нужен только чтобы понять, нужны ли префиксы query:/passage:,
                                                     # т.к. по имени локальной папки MODEL_PATH это не определить

SUMMARIES_PATH = None                  # см. 01_train_biencoder.ipynb - тот же файл, если использовали суммаризацию
TOP_N = 300                            # с запасом (не 50!) - чтобы у LightGBM-ре-ранкера было из чего выбирать

BATCH_SIZE_QUERY = 128   # короткие тексты
BATCH_SIZE_ITEM = 32     # длинные тексты - меньший батч, чтобы не вылетать в shared memory (см. 01)

# режим (2) - benchmark (по умолчанию); для режима (1) - train, раскомментируйте:
QUERIES_FILE = "benchmark_queries.parquet"
ITEMS_FILE = "benchmark_items.parquet"
QUERY_ID_COL = "query_id"
OUT_PATH = "./embed_candidates.parquet"
CACHE_DIR = "./retrieve_cache_benchmark"

# Закомментировать для бенчмарка
# QUERIES_FILE = "train_queries.parquet"
# ITEMS_FILE = "train_items.parquet"
# QUERY_ID_COL = "query_uid"
# OUT_PATH = "./embed_candidates_train.parquet"
# CACHE_DIR = "./retrieve_cache_train"
###

os.makedirs(CACHE_DIR, exist_ok=True)

# Явно выбираем устройство (то же соображение, что в 01_train_biencoder.ipynb):
# encode() и без этого сам использовал бы GPU по умолчанию, но так это видно в выводе.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def is_e5_model(name: str) -> bool:
    return "e5" in name.lower()

## Проверяем, что PyTorch видит GPU

In [12]:
if DEVICE == "cuda":
    print(f"устройство: cuda ({torch.cuda.get_device_name(0)})")
else:
    print("устройство: cpu - torch.cuda.is_available() вернул False (см. подробности в 01_train_biencoder.ipynb)")

устройство: cuda (NVIDIA GeForce RTX 3060 Laptop GPU)


## Загружаем данные и тексты

In [13]:
bq = pd.read_parquet(f"{BASE}/{QUERIES_FILE}")
bi = pd.read_parquet(f"{BASE}/{ITEMS_FILE}")
print(f"queries={len(bq)} items={len(bi)}  (id_col={QUERY_ID_COL})")

summaries = None
if SUMMARIES_PATH:
    summ_df = pd.read_parquet(SUMMARIES_PATH)
    summaries = summ_df.set_index("item_id")["summary"]

e5 = is_e5_model(MODEL_PATH) or is_e5_model(BASE_MODEL_NAME)
q_prefix = "query: " if e5 else ""
p_prefix = "passage: " if e5 else ""

q_text = (q_prefix + build_embedding_query_text(bq)).tolist()
item_summaries_aligned = summaries.reindex(bi["item_id"]).reset_index(drop=True) if summaries is not None else None
it_text = (p_prefix + build_embedding_item_text(bi, use_summary=item_summaries_aligned)).tolist()
q_text[:3]

queries=2452 items=189212  (id_col=query_id)


['query: перевозки владикавказ тбилиси',
 'query: обзвон по базе. Вид услуги Деловые услуги',
 'query: липоредукция подбородка. Вид услуги Красота, здоровье']

## Модель + эмбеддинги объявлений (с кэшем на диск)

In [14]:
model = SentenceTransformer(MODEL_PATH, device=DEVICE, trust_remote_code=True)
model.half()   # fp16 инференс - только forward, безопасно, в разы быстрее и меньше VRAM

# normalize_embeddings=True -> косинусная близость сводится к скалярному
# произведению (IndexFlatIP), что быстрее, чем IndexFlatL2 + отдельная
# нормировка на лету
item_emb_path = os.path.join(CACHE_DIR, "item_emb.npy")
if os.path.exists(item_emb_path):
    item_emb = np.load(item_emb_path)
    print(f"item embeddings загружены из кэша: {item_emb.shape}")
else:
    t0 = time.time()
    item_emb = model.encode(
        it_text, batch_size=BATCH_SIZE_ITEM, show_progress_bar=True,
        normalize_embeddings=True, convert_to_numpy=True,
    ).astype(np.float32)
    print(f"item embeddings: {item_emb.shape} in {time.time()-t0:.1f}s")
    np.save(item_emb_path, item_emb)

Batches: 100%|██████████| 5913/5913 [12:04<00:00,  8.16it/s]


item embeddings: (189212, 768) in 726.3s


## FAISS-индекс (точный brute-force поиск — на 189к x dim это секунды, ANN тут не нужен)

In [15]:
dim = item_emb.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(item_emb)

## Эмбеддинги запросов + поиск top-N (тоже с кэшем на диск)

In [16]:
query_emb_path = os.path.join(CACHE_DIR, "query_emb.npy")
if os.path.exists(query_emb_path):
    q_emb = np.load(query_emb_path)
    print(f"query embeddings загружены из кэша: {q_emb.shape}")
else:
    t0 = time.time()
    q_emb = model.encode(
        q_text, batch_size=BATCH_SIZE_QUERY, show_progress_bar=True,
        normalize_embeddings=True, convert_to_numpy=True,
    ).astype(np.float32)
    print(f"query embeddings: {q_emb.shape} in {time.time()-t0:.1f}s")
    np.save(query_emb_path, q_emb)

idx_path = os.path.join(CACHE_DIR, "search_idx.npy")
scores_path = os.path.join(CACHE_DIR, "search_scores.npy")
if os.path.exists(idx_path) and os.path.exists(scores_path):
    idx = np.load(idx_path)
    scores = np.load(scores_path)
    print("результаты FAISS-поиска загружены из кэша")
else:
    t0 = time.time()
    scores, idx = index.search(q_emb, TOP_N)  # (n_queries, TOP_N) - косинус в [-1, 1]
    print(f"faiss search in {time.time()-t0:.1f}s")
    np.save(idx_path, idx)
    np.save(scores_path, scores)

Batches: 100%|██████████| 20/20 [00:00<00:00, 21.90it/s]


query embeddings: (2452, 768) in 1.0s
faiss search in 2.1s


## Сохраняем кандидатов

Векторизованная сборка (numpy) вместо Python-цикла — тот же цикл на
~354к query x 300 кандидатов строил ~106 млн Python-кортежей и падал по
памяти (`MemoryError`). Здесь то же самое через numpy-массивы - на порядок
дешевле по памяти и на порядки быстрее по времени.

In [17]:
item_ids = bi["item_id"].values
n_queries, top_n = idx.shape

qid_col = bq[QUERY_ID_COL].values
rank_flat = np.tile(np.arange(top_n, dtype=np.int32), n_queries)
query_id_flat = np.repeat(qid_col, top_n)
idx_flat = idx.ravel()
score_flat = scores.ravel()

valid = idx_flat >= 0  # на всякий случай (тут не должно случаться, т.к. items >> TOP_N)

out = pd.DataFrame({
    "query_id": query_id_flat[valid],
    "item_id": item_ids[idx_flat[valid]],
    "embed_score": score_flat[valid].astype(np.float32),
    "rank": rank_flat[valid],
})
out.to_parquet(OUT_PATH, index=False)
print(f"сохранено {OUT_PATH}: {out.shape}")
out.head()

сохранено ./embed_candidates.parquet: (735600, 4)


,query_id,item_id,embed_score,rank
0,70DfDUpwjxB4lzFd,355392014208b7bf,0.741993,0
1,70DfDUpwjxB4lzFd,d722bcda1a555091,0.731867,1
2,70DfDUpwjxB4lzFd,414b206a6ab83aff,0.713573,2
3,70DfDUpwjxB4lzFd,255fbeaf526a1cc1,0.707208,3
4,70DfDUpwjxB4lzFd,6042fecdd7753112,0.703453,4


## (опционально) Посмотреть, что даёт ТОЛЬКО эмбеддинг-канал сам по себе

Без BM25 и без ре-ранкера — топ-k по `embed_score` на запрос. Имеет смысл
только в режиме (2) benchmark (`QUERY_ID_COL == "query_id"`).

In [18]:
def save_answer_csv(candidates_path=OUT_PATH, out_csv="./answer_embed_only.csv", k=50):
    df = pd.read_parquet(candidates_path)
    top = (
        df.sort_values(["query_id", "embed_score"], ascending=[True, False])
        .groupby("query_id")
        .head(k)
    )
    answer = top.groupby("query_id")["item_id"].apply(lambda s: " ".join(s.tolist())).reset_index()
    answer.columns = ["query_id", "answer"]
    answer.to_csv(out_csv, index=False)
    print(f"сохранено {out_csv}: {answer.shape}")


if QUERY_ID_COL == "query_id":
    save_answer_csv()

сохранено ./answer_embed_only.csv: (2452, 2)


## Дальше

Если считали режим (1) train — переходите к `03_train_reranker.ipynb`.
Если режим (2) benchmark — этот файл (`embed_candidates.parquet`) сразу
подхватит `generate_final_answer.py`.

Не забудьте при переключении режима поменять и `CACHE_DIR` (`retrieve_cache_train`
↔ `retrieve_cache_benchmark`), чтобы кэши train и benchmark не путались друг с другом.